In [1]:
pip install sklearn-crfsuite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.0 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


In [15]:
import re

In [2]:
import pandas as pd

file_path = "/kaggle/input/datasets/karimmahmoud09/pii-dataset/transformed_train.parquet"
df = pd.read_parquet(file_path)

df.head()

,masked_text,unmasked_text,token_entity_labels,tokenised_unmasked_text
0,[PREFIX_1] [FIRSTNAME_1] [MIDDLENAME_1] [LASTN...,"Mr. Adolphus Reagan Ziemann, as a Central Prin...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[mr, ., adolph, ##us, reagan, z, ##ie, ##mann,..."
1,"Hello [FIRSTNAME_1], would you please investig...","Hello Hannah, would you please investigate the...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[hello, hannah, ,, would, you, please, investi..."
2,We also request a review of our policies with ...,We also request a review of our policies with ...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[we, also, request, a, review, of, our, polici..."
3,"Dear [FIRSTNAME_1], a company-wide presentatio...","Dear Devan, a company-wide presentation is req...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[dear, dev, ##an, ,, a, company, -, wide, pres..."
4,Can we also have a session on how to manage st...,Can we also have a session on how to manage st...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[can, we, also, have, a, session, on, how, to,..."


In [3]:
sentences = []
s=set()
for _, row in df.iterrows():
    token_entity_labels = row["token_entity_labels"]
    tokenised_unmasked_text = row["tokenised_unmasked_text"]
    for val in token_entity_labels:
        s.add(val)
    sentence = list(zip(tokenised_unmasked_text, token_entity_labels))
    sentences.append(sentence)

In [4]:
def word2features(sent, i):
    word = sent[i][0]

    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[-2:]": word[-2:],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
    }

    # Previous word features
    if i > 0:
        prev_word = sent[i - 1][0]
        features.update({
            "-1:word.lower()": prev_word.lower(),
            "-1:word.istitle()": prev_word.istitle(),
            "-1:word.isupper()": prev_word.isupper(),
        })
    else:
        features["BOS"] = True  # Beginning of sentence

    # Next word features
    if i < len(sent) - 1:
        next_word = sent[i + 1][0]
        features.update({
            "+1:word.lower()": next_word.lower(),
            "+1:word.istitle()": next_word.istitle(),
            "+1:word.isupper()": next_word.isupper(),
        })
    else:
        features["EOS"] = True  

    return features

In [5]:
def extract_features(sentences):
    X = []
    y = []

    for sent in sentences:
        X.append([word2features(sent, i) for i in range(len(sent))])
        y.append([label for (_, label) in sent])

    return X, y

In [6]:
from sklearn.model_selection import train_test_split

X, y = extract_features(sentences)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
import sklearn_crfsuite
from sklearn_crfsuite import metrics

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,   # L1 regularization
    c2=0.1,   # L2 regularization
    max_iterations=100,
    all_possible_transitions=True
)

crf.fit(X_train, y_train)

CRF(algorithm='lbfgs', all_possible_transitions=True, c1=0.1, c2=0.1,
    max_iterations=100)

In [8]:
y_pred = crf.predict(X_test)

print(metrics.flat_classification_report(
    y_test, y_pred, digits=3
))

                    precision    recall  f1-score   support

     B-ACCOUNTNAME      0.981     0.990     0.986       206
   B-ACCOUNTNUMBER      0.679     0.533     0.597       210
B-CREDITCARDNUMBER      0.707     0.624     0.663       205
           B-EMAIL      0.916     0.913     0.915       300
              B-IP      0.455     0.420     0.437       181
            B-IPV4      0.751     0.785     0.768       242
            B-IPV6      0.763     0.755     0.759       196
             B-MAC      0.989     0.995     0.992       189
        B-PASSWORD      0.744     0.459     0.568       209
    B-PHONE_NUMBER      0.970     0.886     0.926       219
             B-SSN      0.970     0.749     0.845       175
        B-USERNAME      0.769     0.538     0.633       279
     I-ACCOUNTNAME      0.981     0.992     0.986       355
   I-ACCOUNTNUMBER      0.569     0.545     0.557       769
I-CREDITCARDNUMBER      0.715     0.650     0.681      1755
           I-EMAIL      0.980     0.990

# additional features

In [9]:
def word2features2(sent, i):
    word = sent[i][0]

    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[-2:]": word[-2:],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
        "word.has_dot()": "." in word,
        "word.has_digit()": any(c.isdigit() for c in word),
        "word.has_colon()": ":" in word,
        "word.is_hex()": bool(re.fullmatch(r"[0-9a-fA-F]+", word)),
    }

    # Previous word features
    if i > 0:
        prev_word = sent[i - 1][0]
        features.update({
            "-1:word.lower()": prev_word.lower(),
            "-1:word.istitle()": prev_word.istitle(),
            "-1:word.isupper()": prev_word.isupper(),
        })
    else:
        features["BOS"] = True  # Beginning of sentence

    # Next word features
    if i < len(sent) - 1:
        next_word = sent[i + 1][0]
        features.update({
            "+1:word.lower()": next_word.lower(),
            "+1:word.istitle()": next_word.istitle(),
            "+1:word.isupper()": next_word.isupper(),
        })
    else:
        features["EOS"] = True  

    return features

In [10]:
def extract_features2(sentences):
    X = []
    y = []

    for sent in sentences:
        X.append([word2features2(sent, i) for i in range(len(sent))])
        y.append([label for (_, label) in sent])

    return X, y

In [16]:
from sklearn.model_selection import train_test_split

X, y = extract_features2(sentences)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [17]:
import sklearn_crfsuite
from sklearn_crfsuite import metrics

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,   # L1 regularization
    c2=0.1,   # L2 regularization
    max_iterations=200,
    all_possible_transitions=True
)

crf.fit(X_train, y_train)

CRF(algorithm='lbfgs', all_possible_transitions=True, c1=0.1, c2=0.1,
    max_iterations=200)

In [18]:
y_pred = crf.predict(X_test)

print(metrics.flat_classification_report(
    y_test, y_pred, digits=3
))

                    precision    recall  f1-score   support

     B-ACCOUNTNAME      0.972     0.995     0.983       206
   B-ACCOUNTNUMBER      0.742     0.562     0.640       210
B-CREDITCARDNUMBER      0.736     0.654     0.693       205
           B-EMAIL      0.920     0.923     0.922       300
              B-IP      0.409     0.370     0.388       181
            B-IPV4      0.741     0.756     0.748       242
            B-IPV6      0.754     0.765     0.759       196
             B-MAC      0.995     1.000     0.997       189
        B-PASSWORD      0.786     0.526     0.630       209
    B-PHONE_NUMBER      0.971     0.909     0.939       219
             B-SSN      0.959     0.806     0.876       175
        B-USERNAME      0.799     0.541     0.645       279
     I-ACCOUNTNAME      0.970     0.997     0.983       355
   I-ACCOUNTNUMBER      0.679     0.563     0.615       769
I-CREDITCARDNUMBER      0.746     0.684     0.714      1755
           I-EMAIL      0.975     0.996